In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

In [ ]:
file_path = "monetary-policy-surprises-data.xlsx"

mps_raw = pd.read_excel(
    file_path,
    sheet_name="FOMC (update 2023)"
)

mps_raw.head()

In [ ]:
mps = mps_raw.copy()

# Make sure Date is a real date
mps["Date"] = pd.to_datetime(mps["Date"])

# Sort by date
mps = mps.sort_values("Date").reset_index(drop=True)

# Keep only 1994–2023
mps_1994_2023 = mps[
    (mps["Date"] >= "1994-01-01") &
    (mps["Date"] <= "2023-12-31")
].copy()

# Baseline sample: scheduled FOMC meetings only
mps_sched = mps_1994_2023[
    mps_1994_2023["Unscheduled"] == 0
].copy()

print("Full 1994–2023 sample:", mps_1994_2023.shape)
print("Scheduled 1994–2023 sample:", mps_sched.shape)
print("Date range:", mps_sched["Date"].min(), "to", mps_sched["Date"].max())

mps_sched.head()

In [ ]:
cols_keep = [
    "Date",
    "Time",
    "Unscheduled",
    "MPS",
    "MPS_ORTH",
    "SP500",
    "SP500 emini",
    "TNOTE02",
    "TNOTE05",
    "TNOTE10"
]

df = mps_sched[cols_keep].copy()
df = df.rename(columns={"SP500 emini": "SP500_emini"})

# NEW: combined S&P 500 series.
# SP500 (large contract) covers 1994 to Dec 2019.
# SP500_emini covers Sep 1997 onward.
# Use SP500 where available, fall back to SP500_emini.
df["SP500_combined"] = df["SP500"].fillna(df["SP500_emini"])

# NEW: yield-curve slope outcomes.
# These test the uncertainty channel directly: HMT (2019) predict text-driven
# news should matter more for the long end, i.e. for the slope rather than the level.
df["SLOPE_10_2"] = df["TNOTE10"] - df["TNOTE02"]
df["SLOPE_10_5"] = df["TNOTE10"] - df["TNOTE05"]
df["SLOPE_5_2"]  = df["TNOTE05"] - df["TNOTE02"]

print(f"Total scheduled events: {len(df)}")
print(f"SP500            non-missing: {df['SP500'].notna().sum()}")
print(f"SP500_emini      non-missing: {df['SP500_emini'].notna().sum()}")
print(f"SP500_combined   non-missing: {df['SP500_combined'].notna().sum()}")
print(f"SP500_combined date range: "
      f"{df.loc[df['SP500_combined'].notna(), 'Date'].min().date()} to "
      f"{df.loc[df['SP500_combined'].notna(), 'Date'].max().date()}")

df.head()

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nSummary statistics:")
print(
    df[[
        "MPS",
        "MPS_ORTH",
        "SP500",
        "SP500_emini",
        "TNOTE02",
        "TNOTE05",
        "TNOTE10"
    ]].describe().T
)

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np

def run_hc3_regression(data, y, x_vars):
    """OLS with HC3 robust standard errors. Drops missing per-regression."""
    reg_data = data[[y] + x_vars].dropna().copy()
    X = sm.add_constant(reg_data[x_vars])
    Y = reg_data[y]
    return sm.OLS(Y, X).fit(cov_type="HC3")

# Expanded outcome list:
#   - Treasury levels (2y, 5y, 10y)
#   - Combined S&P 500 (preferred equity outcome - 239 obs)
#   - Old SP500 and SP500_emini kept for transparency / robustness
#   - Yield-curve slopes (NEW - test of uncertainty channel)
outcomes = [
    "TNOTE02", "TNOTE05", "TNOTE10",
    "SP500_combined",
    "SP500", "SP500_emini",
    "SLOPE_5_2", "SLOPE_10_2", "SLOPE_10_5"
]

baseline_results = {}
for y in outcomes:
    model = run_hc3_regression(df, y, ["MPS"])
    baseline_results[y] = model
    print("\n" + "="*80)
    print(f"Dependent variable: {y}")
    print(model.summary())

In [ ]:
def extract_regression_row(model, outcome, variable="MPS"):
    return {
        "outcome": outcome,
        "coef": model.params.get(variable, np.nan),
        "se_hc3": model.bse.get(variable, np.nan),
        "t_stat": model.tvalues.get(variable, np.nan),
        "p_value": model.pvalues.get(variable, np.nan),
        "r_squared": model.rsquared,
        "n_obs": int(model.nobs)
    }

rows = []

for outcome, model in baseline_results.items():
    rows.append(extract_regression_row(model, outcome, "MPS"))

baseline_table = pd.DataFrame(rows)

baseline_table

In [ ]:
baseline_table_rounded = baseline_table.copy()

for col in ["coef", "se_hc3", "t_stat", "p_value", "r_squared"]:
    baseline_table_rounded[col] = baseline_table_rounded[col].round(4)

baseline_table_rounded

In [ ]:
df.to_csv("mps_clean_scheduled_1994_2023.csv", index=False)

In [ ]:
def significance_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""

table_for_latex = baseline_table.copy()

table_for_latex["coef_se"] = (
    table_for_latex["coef"].map(lambda x: f"{x:.3f}") +
    table_for_latex["p_value"].map(significance_stars) +
    "\n(" +
    table_for_latex["se_hc3"].map(lambda x: f"{x:.3f}") +
    ")"
)

display_table = table_for_latex[["outcome", "coef_se", "r_squared", "n_obs"]].copy()
display_table["r_squared"] = display_table["r_squared"].map(lambda x: f"{x:.3f}")

display_table

In [ ]:
df[["MPS", "TNOTE02", "TNOTE05", "TNOTE10", "SP500", "SP500_emini"]].head(10)

In [ ]:
df[["MPS", "TNOTE02", "TNOTE05", "TNOTE10"]].abs().describe().T

In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://www.federalreserve.gov"
OUT_DIR = Path("fomc_statements")
OUT_DIR.mkdir(exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 academic seminar data collection"
}

def get_soup(url):
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def clean_statement_text(html):
    soup = BeautifulSoup(html, "html.parser")

    # Remove scripts, styles, navigation, footer-like elements
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text("\n")
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]

    # Drop common website/navigation boilerplate
    drop_patterns = [
        r"^Board of Governors",
        r"^The Federal Reserve",
        r"^Federal Open Market Committee",
        r"^Monetary Policy",
        r"^Home$",
        r"^Press releases$",
        r"^Accessibility",
        r"^Contact Us",
        r"^Last update",
        r"^Release Date:",
        r"^For immediate release$",
        r"^Federal Reserve Release$",
        r"^Press Release$",
        r"^Skip to main content",
        r"^Search$",
        r"^Stay Connected",
    ]

    kept = []
    for line in lines:
        if any(re.search(p, line, flags=re.IGNORECASE) for p in drop_patterns):
            continue
        kept.append(line)

    text = "\n".join(kept)

    # Light cleanup
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def extract_date_from_statement_url_or_text(url, text):
    # Try YYYYMMDD in URL
    m = re.search(r"(19|20)\d{6}", url)
    if m:
        return pd.to_datetime(m.group(0), format="%Y%m%d")

    # Try Month DD, YYYY in text
    m = re.search(
        r"(January|February|March|April|May|June|July|August|September|October|November|December)"
        r"\s+\d{1,2},\s+(19|20)\d{2}",
        text
    )
    if m:
        return pd.to_datetime(m.group(0))

    return pd.NaT

def collect_statement_links_from_year(year):
    url = f"{BASE}/monetarypolicy/fomchistorical{year}.htm"
    soup = get_soup(url)

    links = []
    for a in soup.find_all("a"):
        label = a.get_text(" ", strip=True)
        href = a.get("href", "")

        # Keep only actual statement links
        if label.lower() == "statement":
            full_url = urljoin(BASE, href)
            links.append({
                "year": year,
                "label": label,
                "url": full_url
            })

    return links

all_links = []
for year in range(1994, 2024):
    try:
        links = collect_statement_links_from_year(year)
        all_links.extend(links)
        print(year, len(links))
        time.sleep(0.3)
    except Exception as e:
        print("Error for year", year, e)

links_df = pd.DataFrame(all_links).drop_duplicates("url").reset_index(drop=True)
print("Total statement links:", len(links_df))
links_df.head()

In [ ]:
year_counts = links_df.groupby("year").size()
print(year_counts.to_string())

In [ ]:
def collect_statement_links_from_press_year(year):
    url = f"{BASE}/newsevents/pressreleases/{year}-press-fomc.htm"
    soup = get_soup(url)

    links = []
    for a in soup.find_all("a"):
        label = a.get_text(" ", strip=True)
        href = a.get("href", "")

        # We only want the actual FOMC statement press releases
        if "Federal Reserve issues FOMC statement" in label:
            full_url = urljoin(BASE, href)
            links.append({
                "year": year,
                "label": label,
                "url": full_url
            })

    return links


recent_links = []
for year in [2021, 2022, 2023]:
    try:
        links = collect_statement_links_from_press_year(year)
        recent_links.extend(links)
        print(year, len(links))
        time.sleep(0.3)
    except Exception as e:
        print("Error for year", year, e)

recent_links_df = pd.DataFrame(recent_links).drop_duplicates("url").reset_index(drop=True)
recent_links_df

In [ ]:
links_all = pd.concat([links_df, recent_links_df], ignore_index=True)
links_all = links_all.drop_duplicates("url").sort_values(["year", "url"]).reset_index(drop=True)

print("Old links:", len(links_df))
print("Recent links:", len(recent_links_df))
print("Total links:", len(links_all))

print(links_all.groupby("year").size().to_string())

In [ ]:
download_log = []

for _, row in links_all.iterrows():
    url = row["url"]

    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()

        text = clean_statement_text(r.text)
        date = extract_date_from_statement_url_or_text(url, text)

        if pd.isna(date):
            print("Could not infer date:", url)
            download_log.append({"url": url, "date": None, "status": "date_failed"})
            continue

        filename = OUT_DIR / f"{date:%Y-%m-%d}.txt"
        filename.write_text(text, encoding="utf-8")

        download_log.append({
            "url": url,
            "date": f"{date:%Y-%m-%d}",
            "filename": str(filename),
            "n_chars": len(text),
            "status": "ok"
        })

        time.sleep(0.3)

    except Exception as e:
        print("Error:", url, e)
        download_log.append({"url": url, "date": None, "status": f"error: {e}"})

download_log_df = pd.DataFrame(download_log)

print(download_log_df["status"].value_counts())
print("Saved files:", len(list(OUT_DIR.glob("*.txt"))))
download_log_df.head()

In [ ]:
for date in ["1994-02-04", "2000-02-02", "2021-01-27", "2023-12-13"]:
    path = OUT_DIR / f"{date}.txt"
    print("\n" + "="*80)
    print(date, "exists:", path.exists())
    if path.exists():
        txt = path.read_text(encoding="utf-8")
        print(txt[:1000])

In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://www.federalreserve.gov"
OUT_DIR = Path("fomc_statements")
OUT_DIR.mkdir(exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 academic seminar data collection"
}

def extract_statement_body(html):
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    # Try modern Fed press-release content first
    candidates = soup.select(
        "div.col-xs-12.col-sm-8.col-md-8, "
        "div.article, "
        "article, "
        "main"
    )

    paragraphs = []

    for c in candidates:
        ps = c.find_all(["p", "li"])
        candidate_texts = [p.get_text(" ", strip=True) for p in ps]
        candidate_texts = [x for x in candidate_texts if x]

        # Keep the candidate with the most substantial paragraph text
        if len(" ".join(candidate_texts)) > len(" ".join(paragraphs)):
            paragraphs = candidate_texts

    # Fallback for older FRB pages
    if not paragraphs:
        raw_text = soup.get_text("\n")
        paragraphs = [line.strip() for line in raw_text.splitlines() if line.strip()]

    drop_patterns = [
        r"^FRB:",
        r"^Federal Reserve Board",
        r"^Federal Reserve Release",
        r"^Press Release",
        r"^Release Date:",
        r"^For release",
        r"^For immediate release",
        r"^An official website",
        r"^Official websites use",
        r"^Here's how you know",
        r"^\.gov",
        r"^The site is secure",
        r"^https://",
        r"^Submit$",
        r"^Search$",
        r"^Menu$",
        r"^News\s*&\s*Events$",
        r"^PDF$",
        r"^Please enable JavaScript",
        r"^Accessible Version",
        r"^Last Update",
        r"^Last update",
        r"^Home$",
        r"^Share$",
        r"^Print$",
        r"^Email$",
        r"^Back to Top$",
        r"^Board of Governors",
        r"^Federal Open Market Committee$",
        r"^Monetary Policy$",
        r"^Contact$",
        r"^Stay Connected",
        r"^Subscribe",
        r"^\|+$",
        r"^-$",
    ]

    keep = []
    for line in paragraphs:
        line = line.replace("\ufeff", "").replace("ï»¿", "").strip()
        line = re.sub(r"\s+", " ", line)

        if not line:
            continue

        if any(re.search(p, line, flags=re.IGNORECASE) for p in drop_patterns):
            continue

        # Remove voting paragraph because your methodology says voting records are removed
        if re.match(r"^Voting for", line, flags=re.IGNORECASE):
            continue
        if re.match(r"^Voting against", line, flags=re.IGNORECASE):
            continue

        keep.append(line)

    text = "\n\n".join(keep)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_date_from_statement_url_or_text(url, text):
    # Old Fed URLs often contain YYYYMMDD
    m = re.search(r"(19|20)\d{6}", url)
    if m:
        return pd.to_datetime(m.group(0), format="%Y%m%d")

    # Modern URLs often contain monetaryYYYYMMDDa.htm
    m = re.search(r"monetary((19|20)\d{6})", url)
    if m:
        return pd.to_datetime(m.group(1), format="%Y%m%d")

    # Fallback: Month DD, YYYY in text
    m = re.search(
        r"(January|February|March|April|May|June|July|August|September|October|November|December)"
        r"\s+\d{1,2},\s+(19|20)\d{2}",
        text
    )
    if m:
        return pd.to_datetime(m.group(0))

    return pd.NaT

In [ ]:
download_log = []

for _, row in links_all.iterrows():
    url = row["url"]

    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()

        text = extract_statement_body(r.text)
        date = extract_date_from_statement_url_or_text(url, text)

        if pd.isna(date):
            print("Could not infer date:", url)
            download_log.append({
                "url": url,
                "date": None,
                "status": "date_failed"
            })
            continue

        filename = OUT_DIR / f"{date:%Y-%m-%d}.txt"
        filename.write_text(text, encoding="utf-8")

        download_log.append({
            "url": url,
            "date": f"{date:%Y-%m-%d}",
            "filename": str(filename),
            "n_chars": len(text),
            "status": "ok"
        })

        time.sleep(0.25)

    except Exception as e:
        print("Error:", url, e)
        download_log.append({
            "url": url,
            "date": None,
            "status": f"error: {e}"
        })

download_log_df = pd.DataFrame(download_log)

print(download_log_df["status"].value_counts())
print("Saved files:", len(list(OUT_DIR.glob("*.txt"))))
download_log_df.head()

In [ ]:
for date in ["1994-02-04", "2000-02-02", "2021-01-27", "2023-12-13"]:
    path = OUT_DIR / f"{date}.txt"
    print("\n" + "="*80)
    print(date, "exists:", path.exists())
    if path.exists():
        txt = path.read_text(encoding="utf-8")
        print("Characters:", len(txt))
        print(txt[:1500])

In [ ]:
statement_rows = []

for file in sorted(OUT_DIR.glob("*.txt")):
    text = file.read_text(encoding="utf-8")
    statement_rows.append({
        "Date": pd.to_datetime(file.stem),
        "filename": file.name,
        "n_chars": len(text),
        "text_start": text[:120]
    })

statement_check = pd.DataFrame(statement_rows).sort_values("Date")

print(statement_check.shape)
statement_check.sort_values("n_chars").head(20)

In [ ]:
from pathlib import Path
import re

OUT_DIR = Path("fomc_statements")

extra_drop_patterns = [
    r"^Accessibility$",
    r"^Contact Us$",
    r"^Careers$",
    r"^FOIA$",
    r"^No FEAR Act$",
    r"^Privacy$",
    r"^Open Government$",
    r"^Plain Writing$",
    r"^USA\.gov$",
    r"^\.\.\.$",
    r"^\|+$"
]

def final_clean_saved_text(text):
    lines = [line.strip() for line in text.splitlines()]
    keep = []

    for line in lines:
        line = line.replace("\ufeff", "").replace("ï»¿", "").strip()
        line = re.sub(r"\s+", " ", line)

        if not line:
            continue

        if any(re.search(p, line, flags=re.IGNORECASE) for p in extra_drop_patterns):
            continue

        keep.append(line)

    return "\n\n".join(keep).strip()

for file in OUT_DIR.glob("*.txt"):
    text = file.read_text(encoding="utf-8")
    cleaned = final_clean_saved_text(text)
    file.write_text(cleaned, encoding="utf-8")

print("Final cleanup done.")

In [ ]:
for date in ["1994-02-04", "2000-02-02", "2021-01-27", "2023-12-13"]:
    path = OUT_DIR / f"{date}.txt"
    print("\n" + "="*80)
    print(date, "exists:", path.exists())
    if path.exists():
        txt = path.read_text(encoding="utf-8")
        print("Characters:", len(txt))
        print(txt[:1200])

In [ ]:
for date in ["2021-01-27", "2023-12-13"]:
    path = OUT_DIR / f"{date}.txt"
    txt = path.read_text(encoding="utf-8")
    
    print("\n" + "="*80)
    print(date)
    print("Characters:", len(txt))
    print("First 800 characters:")
    print(txt[:800])
    print("\nLast 800 characters:")
    print(txt[-800:])

In [ ]:
suspicious = []

for file in sorted(OUT_DIR.glob("*.txt")):
    txt = file.read_text(encoding="utf-8")
    if "\n...\n" in txt or txt.strip() == "...":
        suspicious.append(file.name)

print("Files containing standalone ellipsis:", len(suspicious))
print(suspicious[:30])

In [ ]:
import re
from pathlib import Path

OUT_DIR = Path("fomc_statements")

def clean_remaining_boilerplate(text):
    lines = [line.strip() for line in text.splitlines()]
    keep = []

    drop_patterns = [
        r"^\.\.\.$",
        r"^For media inquiries",
        r"^Implementation Note issued",
        r"^Implementation note issued",
        r"^Implementation Note$",
        r"^Minutes of the Federal Open Market Committee",
        r"^Statement on Longer-Run Goals",
        r"^PDF$",
        r"^Accessible Version$",
        r"^Page not found",
    ]

    for line in lines:
        line = line.replace("\ufeff", "").replace("ï»¿", "").strip()
        line = re.sub(r"\s+", " ", line)

        if not line:
            continue

        if any(re.search(p, line, flags=re.IGNORECASE) for p in drop_patterns):
            continue

        keep.append(line)

    return "\n\n".join(keep).strip()


for file in OUT_DIR.glob("*.txt"):
    text = file.read_text(encoding="utf-8")
    cleaned = clean_remaining_boilerplate(text)
    file.write_text(cleaned, encoding="utf-8")

print("Remaining boilerplate cleanup done.")

In [ ]:
for date in ["2021-01-27", "2023-12-13"]:
    path = OUT_DIR / f"{date}.txt"
    txt = path.read_text(encoding="utf-8")
    
    print("\n" + "="*80)
    print(date)
    print("Characters:", len(txt))
    print("First 700 characters:")
    print(txt[:700])
    print("\nLast 700 characters:")
    print(txt[-700:])

In [ ]:
suspicious_terms = [
    "...",
    "For media inquiries",
    "Implementation Note issued",
    "Federal Reserve Board -",
    "An official website",
    "Please enable JavaScript",
    "Accessibility",
    "Contact Us"
]

for term in suspicious_terms:
    bad_files = []
    for file in sorted(OUT_DIR.glob("*.txt")):
        txt = file.read_text(encoding="utf-8")
        if term in txt:
            bad_files.append(file.name)

    print(term, ":", len(bad_files))
    if bad_files:
        print(bad_files[:20])

In [ ]:
statement_rows = []

for file in sorted(OUT_DIR.glob("*.txt")):
    text = file.read_text(encoding="utf-8")

    statement_rows.append({
        "Date": pd.to_datetime(file.stem),
        "statement_text": text,
        "n_chars": len(text),
        "n_words": len(text.split())
    })

statements = (
    pd.DataFrame(statement_rows)
    .sort_values("Date")
    .reset_index(drop=True)
)

merged = df.merge(statements, on="Date", how="inner")

print("Market rows:", df.shape[0])
print("Statement rows:", statements.shape[0])
print("Merged rows:", merged.shape[0])

print("\nMerged sample range:")
print(merged["Date"].min(), "to", merged["Date"].max())

print("\nFirst rows:")
print(merged[[
    "Date", "MPS", "TNOTE02", "TNOTE05", "TNOTE10",
    "SP500", "n_chars", "n_words"
]].head())

In [ ]:
market_dates = set(df["Date"])
statement_dates = set(statements["Date"])

missing_statements = sorted(market_dates - statement_dates)
extra_statements = sorted(statement_dates - market_dates)

print("Market dates without statement:", len(missing_statements))
print(missing_statements[:40])

print("\nStatement dates without scheduled market row:", len(extra_statements))
print(extra_statements[:40])

In [ ]:
merged.to_csv("fomc_market_text_merged_raw.csv", index=False)
statements.to_csv("fomc_statements_clean.csv", index=False)

In [ ]:
# Inspect the suspicious 2007-06-18 file
path = OUT_DIR / "2007-06-18.txt"

if path.exists():
    txt = path.read_text(encoding="utf-8")
    print("Characters:", len(txt))
    print(txt[:1500])
else:
    print("2007-06-18.txt does not exist.")

In [ ]:
links_all[links_all["year"] == 2007][["label", "url"]].to_string(index=False)

In [ ]:
from pathlib import Path
Path("fomc_statements/2007-06-18.txt").unlink()
print("Deleted the stray duplicate. Corpus is clean.")

In [ ]:
from pathlib import Path
dates = sorted(p.stem for p in Path("fomc_statements").glob("*.txt"))
merged_dates = set(merged_finbert["Date"].dt.strftime("%Y-%m-%d"))
orphans = [d for d in dates if d not in merged_dates]
print("corpus files:", len(dates), "| files not in merged data:", orphans)

In [ ]:
print("2007-06-18 in merged?", (merged_finbert["Date"] == "2007-06-18").any())
print("2007-06-28 in merged?", (merged_finbert["Date"] == "2007-06-28").any())

In [ ]:
statement_rows = []

for file in sorted(OUT_DIR.glob("*.txt")):
    text = file.read_text(encoding="utf-8")

    statement_rows.append({
        "Date": pd.to_datetime(file.stem),
        "statement_text": text,
        "n_chars": len(text),
        "n_words": len(text.split())
    })

statements = (
    pd.DataFrame(statement_rows)
    .sort_values("Date")
    .reset_index(drop=True)
)

merged = df.merge(statements, on="Date", how="inner")

print("Market rows:", df.shape[0])
print("Statement rows:", statements.shape[0])
print("Merged rows:", merged.shape[0])

market_dates = set(df["Date"])
statement_dates = set(statements["Date"])

missing_statements = sorted(market_dates - statement_dates)
extra_statements = sorted(statement_dates - market_dates)

print("\nMarket dates without statement:", len(missing_statements))
print(missing_statements[:40])

print("\nStatement dates without scheduled market row:", len(extra_statements))
print(extra_statements[:40])

In [ ]:
import re
import pandas as pd
import numpy as np

# Each entry is (phrase, polarity) where polarity is +1 hawkish, -1 dovish.
# Phrases are matched longest-first with masking so multi-word terms preempt
# their substrings. Example: 'removing accommodation' is counted as +1 hawkish
# without also firing 'accommodation' as -1 dovish.
#
# This list improves on the previous version by:
#   (i)   removing brittle standalone words ('lower', 'unemployment', 'firm',
#         'price stability') that systematically misclassified hawkish text.
#   (ii)  adding standard FOMC forward-guidance phrases ('patient',
#         'considerable period', 'considerable time') as dovish signals.
#   (iii) adding QT/restrictive language used in the 2022-2023 tightening cycle.
policy_terms = [
    # ---- Hawkish: tightening / removing accommodation ----
    ("policy firming", +1),
    ("additional policy firming", +1),
    ("further policy firming", +1),
    ("removing policy accommodation", +1),
    ("removing accommodation", +1),
    ("remove accommodation", +1),
    ("reduce policy accommodation", +1),
    ("reduce accommodation", +1),
    ("reduced accommodation", +1),
    ("less accommodative", +1),
    ("further increases", +1),
    ("further tightening", +1),
    ("ongoing increases", +1),
    ("ongoing tightening", +1),
    ("additional firming", +1),
    ("inflation pressures", +1),
    ("inflation pressure", +1),
    ("inflation risks", +1),
    ("inflation risk", +1),
    ("elevated inflation", +1),
    ("persistent inflation", +1),
    ("inflation remains elevated", +1),
    ("sufficiently restrictive", +1),
    ("restrictive stance", +1),
    ("restrictive", +1),
    ("overheating", +1),
    ("tighten", +1),
    ("tightening", +1),
    ("tightened", +1),
    ("firming", +1),
    ("raise the target", +1),
    ("raised the target", +1),
    ("raises the target", +1),
    ("raising the target", +1),
    ("increase the target", +1),
    ("increased the target", +1),
    ("increases in the target", +1),
    ("hike", +1),
    ("hikes", +1),

    # ---- Dovish: easing / accommodation / forward-guidance dovish ----
    ("provide additional accommodation", -1),
    ("highly accommodative", -1),
    ("considerable period", -1),
    ("considerable time", -1),
    ("patient",  -1),
    ("prepared to adjust", -1),
    ("support the economy", -1),
    ("support economic activity", -1),
    ("downside risks", -1),
    ("downside risk", -1),
    ("subdued inflation", -1),
    ("below the committee", -1),
    ("below its longer-run objective", -1),
    ("substantial further progress", -1),
    ("accommodative stance", -1),
    ("accommodative",  -1),
    ("ease the stance", -1),
    ("easing", -1),
    ("eased", -1),
    ("ease", -1),
    ("lower the target", -1),
    ("lowered the target", -1),
    ("reduce the target", -1),
    ("reduced the target", -1),
    ("cut the target", -1),
    ("cuts", -1),
    ("rate cuts", -1),
    ("weakness", -1),
    ("weaker", -1),
    ("slowdown", -1),
    ("slowing", -1),
    ("recession", -1),
    ("job losses", -1),
    ("strains in financial markets", -1),
    ("financial market strains", -1),
]

assert len(set(p for p,_ in policy_terms)) == len(policy_terms), "Duplicate phrase!"
n_hawk = sum(1 for _,p in policy_terms if p > 0)
n_dov  = sum(1 for _,p in policy_terms if p < 0)
print(f"Loaded dictionary: {n_hawk} hawkish phrases, {n_dov} dovish phrases.")

In [ ]:
def normalize_text(text):
    text = str(text).lower()
    text = text.replace("\u2019", "'")
    text = re.sub(r"[^a-z0-9\s\-']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def count_terms_masked(text, terms_with_polarity):
    """
    Count hawkish and dovish hits with longest-first masking so that
    multi-word phrases preempt their substrings.

    Example: 'removing accommodation' is counted as +1 hawkish and does
    NOT also fire 'accommodation' as -1 dovish, because the multi-word
    phrase matches first and masks its tokens before the standalone
    word can be matched.
    """
    text_norm = normalize_text(text)
    terms_sorted = sorted(
        terms_with_polarity,
        key=lambda x: -len(x[0].split())
    )
    hawkish_count = 0
    dovish_count = 0

    for phrase, polarity in terms_sorted:
        pattern = r"\b" + re.escape(normalize_text(phrase)) + r"\b"
        matches = re.findall(pattern, text_norm)
        n = len(matches)
        if n == 0:
            continue
        if polarity > 0:
            hawkish_count += n
        else:
            dovish_count += n
        text_norm = re.sub(pattern, " __MSK__ ", text_norm)

    return hawkish_count, dovish_count


def compute_bow_score(text):
    H, D = count_terms_masked(text, policy_terms)
    score = (H - D) / (H + D + 1)
    return pd.Series({
        "hawkish_count": H,
        "dovish_count": D,
        "BoW": score
    })


# Sanity checks against the failure modes of the previous dictionary
print("Dictionary sanity tests:")
for label, txt in [
    ("hawkish: 'removing accommodation'",
     "The Committee is removing accommodation by raising the target range."),
    ("dovish: forward-guidance 'patient'",
     "The Committee will be patient in deciding when to begin removing policy accommodation."),
    ("ambiguous (should not score)",
     "Inflation has moved lower but the labor market remains tight."),
]:
    H, D = count_terms_masked(txt, policy_terms)
    print(f"  {label:55s} -> H={H}, D={D}")

In [ ]:
merged_bow = merged.copy()

bow_scores = merged_bow["statement_text"].apply(compute_bow_score)

merged_bow = pd.concat([merged_bow, bow_scores], axis=1)

merged_bow = merged_bow.sort_values("Date").reset_index(drop=True)
merged_bow["d_BoW"] = merged_bow["BoW"].diff()

merged_bow[[
    "Date", "hawkish_count", "dovish_count", "BoW", "d_BoW", "n_words"
]].head(15)

In [ ]:
merged_bow[["BoW", "d_BoW", "hawkish_count", "dovish_count"]].describe().T

In [ ]:
print("Most hawkish statements:")
display(
    merged_bow.sort_values("BoW", ascending=False)[
        ["Date", "BoW", "hawkish_count", "dovish_count", "n_words"]
    ].head(10)
)

print("Most dovish statements:")
display(
    merged_bow.sort_values("BoW", ascending=True)[
        ["Date", "BoW", "hawkish_count", "dovish_count", "n_words"]
    ].head(10)
)

In [ ]:
outcomes = [
    "TNOTE02", "TNOTE05", "TNOTE10",
    "SP500_combined",
    "SP500", "SP500_emini",
    "SLOPE_5_2", "SLOPE_10_2", "SLOPE_10_5"
]

bow_results = {}
for y in outcomes:
    model = run_hc3_regression(merged_bow, y, ["MPS", "d_BoW"])
    bow_results[y] = model
    print("\n" + "="*80)
    print(f"Dependent variable: {y}")
    print(model.summary())

In [ ]:
def extract_two_var_result(model, outcome):
    return {
        "outcome": outcome,
        "MPS_coef": model.params.get("MPS", np.nan),
        "MPS_se": model.bse.get("MPS", np.nan),
        "MPS_p": model.pvalues.get("MPS", np.nan),
        "d_BoW_coef": model.params.get("d_BoW", np.nan),
        "d_BoW_se": model.bse.get("d_BoW", np.nan),
        "d_BoW_p": model.pvalues.get("d_BoW", np.nan),
        "r_squared": model.rsquared,
        "n_obs": int(model.nobs)
    }

bow_table = pd.DataFrame([
    extract_two_var_result(model, outcome)
    for outcome, model in bow_results.items()
])

bow_table.round(4)

In [ ]:
merged_bow.to_csv("fomc_market_text_bow.csv", index=False)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm
import re
import numpy as np
import pandas as pd

In [ ]:
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.eval()

In [ ]:
model.config.id2label

In [ ]:


def split_into_sentences(text):
    """
    Simple sentence splitter for FOMC statements.
    Keeps the approach transparent and reproducible.
    """
    text = str(text).replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    
    # Split after sentence-ending punctuation followed by space and capital letter
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)
    
    # Remove very short fragments
    sentences = [s.strip() for s in sentences if len(s.strip().split()) >= 4]
    
    return sentences

In [ ]:
test_sentences = split_into_sentences(merged_bow.loc[0, "statement_text"])

print("Number of sentences:", len(test_sentences))
for s in test_sentences[:5]:
    print("-", s)

In [ ]:
def score_sentence_finbert(sentence, tokenizer, model):
    """
    Returns positive, negative, and neutral probabilities for one sentence.
    """
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    
    return {
        "p_positive": float(probs[0]),
        "p_negative": float(probs[1]),
        "p_neutral": float(probs[2])
    }

In [ ]:
example_sentence = test_sentences[0]
print(example_sentence)

score_sentence_finbert(example_sentence, tokenizer, model)

In [ ]:
def score_statement_finbert(text, tokenizer, model):
    """
    Applies FinBERT sentence by sentence and aggregates to document level.
    FinBERT score = average positive probability minus average negative probability.
    """
    sentences = split_into_sentences(text)
    
    if len(sentences) == 0:
        return pd.Series({
            "FinBERT": np.nan,
            "finbert_pos": np.nan,
            "finbert_neg": np.nan,
            "finbert_neu": np.nan,
            "finbert_n_sentences": 0
        })
    
    scores = [
        score_sentence_finbert(sentence, tokenizer, model)
        for sentence in sentences
    ]
    
    scores_df = pd.DataFrame(scores)
    
    pos = scores_df["p_positive"].mean()
    neg = scores_df["p_negative"].mean()
    neu = scores_df["p_neutral"].mean()
    
    finbert_score = pos - neg
    
    return pd.Series({
        "FinBERT": finbert_score,
        "finbert_pos": pos,
        "finbert_neg": neg,
        "finbert_neu": neu,
        "finbert_n_sentences": len(sentences)
    })

In [ ]:
score_statement_finbert(
    merged_bow.loc[0, "statement_text"],
    tokenizer,
    model
)

In [ ]:
tqdm.pandas()

finbert_scores = merged_bow["statement_text"].progress_apply(
    lambda x: score_statement_finbert(x, tokenizer, model)
)

merged_finbert = pd.concat([merged_bow, finbert_scores], axis=1)

merged_finbert = merged_finbert.sort_values("Date").reset_index(drop=True)

merged_finbert["d_FinBERT"] = merged_finbert["FinBERT"].diff()

merged_finbert[[
    "Date",
    "FinBERT",
    "d_FinBERT",
    "finbert_pos",
    "finbert_neg",
    "finbert_neu",
    "finbert_n_sentences"
]].head(15)

In [ ]:
merged_finbert[[
    "FinBERT",
    "d_FinBERT",
    "finbert_pos",
    "finbert_neg",
    "finbert_neu",
    "finbert_n_sentences"
]].describe().T

In [ ]:
print("Most positive FinBERT statements:")
display(
    merged_finbert.sort_values("FinBERT", ascending=False)[
        ["Date", "FinBERT", "finbert_pos", "finbert_neg", "finbert_neu", "finbert_n_sentences", "n_words"]
    ].head(10)
)

print("Most negative FinBERT statements:")
display(
    merged_finbert.sort_values("FinBERT", ascending=True)[
        ["Date", "FinBERT", "finbert_pos", "finbert_neg", "finbert_neu", "finbert_n_sentences", "n_words"]
    ].head(10)
)

In [ ]:
finbert_results = {}
for y in outcomes:
    model = run_hc3_regression(merged_finbert, y, ["MPS", "d_FinBERT"])
    finbert_results[y] = model
    print("\n" + "="*80)
    print(f"Dependent variable: {y}")
    print(model.summary())

In [ ]:
def extract_finbert_result(model, outcome):
    return {
        "outcome": outcome,
        "MPS_coef": model.params.get("MPS", np.nan),
        "MPS_se": model.bse.get("MPS", np.nan),
        "MPS_p": model.pvalues.get("MPS", np.nan),
        "d_FinBERT_coef": model.params.get("d_FinBERT", np.nan),
        "d_FinBERT_se": model.bse.get("d_FinBERT", np.nan),
        "d_FinBERT_p": model.pvalues.get("d_FinBERT", np.nan),
        "r_squared": model.rsquared,
        "n_obs": int(model.nobs)
    }

finbert_table = pd.DataFrame([
    extract_finbert_result(model, outcome)
    for outcome, model in finbert_results.items()
])

finbert_table.round(4)

In [ ]:
merged_finbert.to_csv("fomc_market_text_bow_finbert.csv", index=False)

In [ ]:
combined_results = {}
for y in outcomes:
    model = run_hc3_regression(
        merged_finbert, y, ["MPS", "d_BoW", "d_FinBERT"]
    )
    combined_results[y] = model
    print("\n" + "="*80)
    print(f"Dependent variable: {y}")
    print(model.summary())

In [ ]:
# Diagnostic: correlation of text shocks with MPS.
# If text shocks are highly correlated with MPS, then including MPS as a
# control absorbs the text variation and could explain a partial null result.
# If text shocks are essentially orthogonal to MPS, then a null result is a
# genuine finding that text does not predict yields beyond what is already
# in the short-rate futures.

print("Correlations of text variables with MPS (full text sample):")
for v in ["BoW", "FinBERT", "d_BoW", "d_FinBERT"]:
    sub = merged_finbert[[v, "MPS"]].dropna()
    r = sub.corr().iloc[0, 1]
    print(f"  Corr({v:<10}, MPS) = {r:+.4f}  (N={len(sub)})")

In [ ]:
# Compact summary tables — one row per outcome.

def extract(model, var):
    if var not in model.params.index:
        return {"coef": np.nan, "se": np.nan, "p": np.nan}
    return {
        "coef": model.params[var],
        "se":   model.bse[var],
        "p":    model.pvalues[var],
    }

def star(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""

def summarize(results_dict, vars_to_show):
    rows = []
    for outcome, m in results_dict.items():
        row = {"outcome": outcome, "N": int(m.nobs), "R2": round(m.rsquared, 4)}
        for v in vars_to_show:
            e = extract(m, v)
            if not np.isnan(e["coef"]):
                row[f"{v}"]   = f"{e['coef']:+.4f}{star(e['p'])}"
                row[f"{v}_se"] = f"({e['se']:.4f})"
                row[f"{v}_p"]  = round(e["p"], 4)
        rows.append(row)
    return pd.DataFrame(rows)

print("="*90)
print("BASELINE: MPS only")
print("="*90)
print(summarize(baseline_results, ["MPS"]).to_string(index=False))

print("\n" + "="*90)
print("BoW + MPS")
print("="*90)
print(summarize(bow_results, ["MPS", "d_BoW"]).to_string(index=False))

print("\n" + "="*90)
print("FinBERT + MPS")
print("="*90)
print(summarize(finbert_results, ["MPS", "d_FinBERT"]).to_string(index=False))

print("\n" + "="*90)
print("COMBINED: MPS + d_BoW + d_FinBERT")
print("="*90)
print(summarize(combined_results, ["MPS", "d_BoW", "d_FinBERT"]).to_string(index=False))

In [ ]:
def extract_combined_result(model, outcome):
    return {
        "outcome": outcome,
        "MPS_coef": model.params.get("MPS", np.nan),
        "MPS_se": model.bse.get("MPS", np.nan),
        "MPS_p": model.pvalues.get("MPS", np.nan),
        "d_BoW_coef": model.params.get("d_BoW", np.nan),
        "d_BoW_se": model.bse.get("d_BoW", np.nan),
        "d_BoW_p": model.pvalues.get("d_BoW", np.nan),
        "d_FinBERT_coef": model.params.get("d_FinBERT", np.nan),
        "d_FinBERT_se": model.bse.get("d_FinBERT", np.nan),
        "d_FinBERT_p": model.pvalues.get("d_FinBERT", np.nan),
        "r_squared": model.rsquared,
        "n_obs": int(model.nobs)
    }

combined_table = pd.DataFrame([
    extract_combined_result(model, outcome)
    for outcome, model in combined_results.items()
])

combined_table.round(4)

In [ ]:
# Batch 2: HMT-style residualization of text scores against macro pre-state.
#
# Hansen, McMahon, and Tong (2019) residualize their text-based topic measures
# against the central bank's numerical forecast variables, treating the residual
# as the "narrative shock" - i.e. the part of the text not predictable from the
# macro state. The Bauer-Swanson dataset already provides four pre-meeting macro
# variables (SP500_3M, SLOPE_3M, BCOM_3M, NFP_12M) that play an analogous role.

# Re-read raw B-S file to get macro columns
bs_with_macro = pd.read_excel(
    "monetary-policy-surprises-data.xlsx",
    sheet_name="FOMC (update 2023)"
)
bs_with_macro["Date"] = pd.to_datetime(bs_with_macro["Date"])

macro_cols = ["Date", "SP500_3M", "SLOPE_3M", "BCOM_3M", "NFP_12M"]
merged_hmt = merged_finbert.merge(
    bs_with_macro[macro_cols],
    on="Date",
    how="left"
)

print("Macro variable coverage in text-regression sample:")
for c in ["SP500_3M", "SLOPE_3M", "BCOM_3M", "NFP_12M"]:
    print(f"  {c}: {merged_hmt[c].notna().sum()}/{len(merged_hmt)} non-missing")

In [ ]:
# Residualize each text measure (level and change) against the macro pre-state.
# The residual is the "narrative shock" - the variation in text that is not
# predictable from publicly known macro/financial state at the time of the meeting.

macro_vars = ["SP500_3M", "SLOPE_3M", "BCOM_3M", "NFP_12M"]

def residualize(data, y, controls):
    """Return Series of OLS residuals aligned with data's index."""
    sub = data[[y] + controls].dropna()
    X = sm.add_constant(sub[controls])
    Y = sub[y]
    m = sm.OLS(Y, X).fit()
    res = pd.Series(np.nan, index=data.index)
    res.loc[sub.index] = m.resid
    return res, m

print("Step 1: regress text measures on macro pre-state\n")

merged_hmt["BoW_res"], m1 = residualize(merged_hmt, "BoW", macro_vars)
print(f"BoW (level) on macro:        R^2 = {m1.rsquared:.4f}, N = {int(m1.nobs)}")

merged_hmt["FinBERT_res"], m2 = residualize(merged_hmt, "FinBERT", macro_vars)
print(f"FinBERT (level) on macro:    R^2 = {m2.rsquared:.4f}, N = {int(m2.nobs)}")

merged_hmt["d_BoW_res"], m3 = residualize(merged_hmt, "d_BoW", macro_vars)
print(f"d_BoW (change) on macro:     R^2 = {m3.rsquared:.4f}, N = {int(m3.nobs)}")

merged_hmt["d_FinBERT_res"], m4 = residualize(merged_hmt, "d_FinBERT", macro_vars)
print(f"d_FinBERT (change) on macro: R^2 = {m4.rsquared:.4f}, N = {int(m4.nobs)}")

print("\nNote: low R^2 in Step 1 means most variation in text is NOT predictable")
print("from macro pre-state, so residualization preserves most of the signal.")

In [ ]:
# Step 2: regress asset-price responses on MPS plus the residualized text shocks.
# If text genuinely carries incremental information beyond MPS and macro pre-state,
# the residualized text coefficients should be significant - especially at longer
# maturities (the uncertainty channel prediction).

outcomes_hmt = [
    "TNOTE02", "TNOTE05", "TNOTE10",
    "SP500_combined",
    "SLOPE_5_2", "SLOPE_10_2", "SLOPE_10_5"
]

print("="*100)
print("HMT-STYLE TABLE A: MPS + residualized level BoW")
print("="*100)
hmt_a = {}
for y in outcomes_hmt:
    hmt_a[y] = run_hc3_regression(merged_hmt, y, ["MPS", "BoW_res"])

print(summarize(hmt_a, ["MPS", "BoW_res"]).to_string(index=False))

print("\n" + "="*100)
print("HMT-STYLE TABLE B: MPS + residualized level FinBERT")
print("="*100)
hmt_b = {}
for y in outcomes_hmt:
    hmt_b[y] = run_hc3_regression(merged_hmt, y, ["MPS", "FinBERT_res"])
print(summarize(hmt_b, ["MPS", "FinBERT_res"]).to_string(index=False))

print("\n" + "="*100)
print("HMT-STYLE TABLE C: MPS + residualized CHANGES (d_BoW_res + d_FinBERT_res)")
print("="*100)
hmt_c = {}
for y in outcomes_hmt:
    hmt_c[y] = run_hc3_regression(merged_hmt, y, ["MPS", "d_BoW_res", "d_FinBERT_res"])
print(summarize(hmt_c, ["MPS", "d_BoW_res", "d_FinBERT_res"]).to_string(index=False))

print("\n" + "="*100)
print("HMT-STYLE TABLE D: MPS + LEVELS joint (BoW_res + FinBERT_res)")
print("="*100)
hmt_d = {}
for y in outcomes_hmt:
    hmt_d[y] = run_hc3_regression(merged_hmt, y, ["MPS", "BoW_res", "FinBERT_res"])
print(summarize(hmt_d, ["MPS", "BoW_res", "FinBERT_res"]).to_string(index=False))

# Diagnostic
print("\n" + "="*100)
print("DIAGNOSTIC: orthogonality of residualized shocks to MPS")
print("="*100)
for v in ["BoW_res", "FinBERT_res", "d_BoW_res", "d_FinBERT_res"]:
    sub = merged_hmt[[v, "MPS"]].dropna()
    r = sub.corr().iloc[0, 1]
    print(f"  Corr({v:<14}, MPS) = {r:+.4f}  (N = {len(sub)})")

In [ ]:
orth_results = {}

for y in outcomes:
    model = run_hc3_regression(
        merged_finbert,
        y,
        ["MPS_ORTH", "d_FinBERT"]
    )
    orth_results[y] = model
    
    print("\n" + "="*80)
    print(f"Dependent variable: {y}")
    print(model.summary())

In [ ]:
def extract_orth_result(model, outcome):
    return {
        "outcome": outcome,
        "MPS_ORTH_coef": model.params.get("MPS_ORTH", np.nan),
        "MPS_ORTH_se": model.bse.get("MPS_ORTH", np.nan),
        "MPS_ORTH_p": model.pvalues.get("MPS_ORTH", np.nan),
        "d_FinBERT_coef": model.params.get("d_FinBERT", np.nan),
        "d_FinBERT_se": model.bse.get("d_FinBERT", np.nan),
        "d_FinBERT_p": model.pvalues.get("d_FinBERT", np.nan),
        "r_squared": model.rsquared,
        "n_obs": int(model.nobs)
    }

orth_table = pd.DataFrame([
    extract_orth_result(model, outcome)
    for outcome, model in orth_results.items()
])

orth_table.round(4)

In [ ]:
merged_finbert[["BoW", "FinBERT", "d_BoW", "d_FinBERT"]].corr()

In [ ]:
no_covid = merged_finbert[
    ~(
        (merged_finbert["Date"] >= "2020-03-01") &
        (merged_finbert["Date"] <= "2020-12-31")
    )
].copy()

no_covid_results = {}

for y in outcomes:
    model = run_hc3_regression(
        no_covid,
        y,
        ["MPS", "d_FinBERT"]
    )
    no_covid_results[y] = model

no_covid_table = pd.DataFrame([
    extract_finbert_result(model, outcome)
    for outcome, model in no_covid_results.items()
])

no_covid_table.round(4)

In [ ]:
# ######################################################################
# RESUME EXTENSIONS (added): LDA topic model + transformer measures
#   Run AFTER the existing cells (needs merged_finbert, merged_hmt,
#   run_hc3_regression, residualize, summarize/star, split_into_sentences,
#   normalize_text, outcomes, outcomes_hmt, macro_vars).
#   New deps: gensim, scikit-learn  (transformers/torch already used).
# ######################################################################


In [ ]:
# ======================================================================
# EXTENSION 1 - LDA TOPIC MODELLING (scikit-learn) - with boilerplate stripping
# ======================================================================
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation

# Administrative / scraped-chrome sentences to drop BEFORE topic modelling.
# These are repeated scaffolding (discount-rate action, voting record, the
# post-2008 implementation note, website nav) that otherwise form their own
# "topics" and crowd out the economic content.
ADMIN_PAT = re.compile(
    r"\b(in a related action|board of governors|boards of directors|"
    r"federal reserve banks?|requests submitted|discount rate|primary credit rate|"
    r"reverse repurchase|reserve balances|interest on reserve|voting (for|against)|"
    r"news\s*&?\s*events|media inquiries|last update|recent postings|implementation note)\b",
    re.I)

def strip_boilerplate(text):
    # everything after this header is the administrative implementation note
    t = re.split(r"Decisions Regarding Monetary Policy Implementation", str(text))[0]
    sents = re.split(r"(?<=[.!?])\s+", t)
    return " ".join(s for s in sents if not ADMIN_PAT.search(s))

FOMC_BOILERPLATE = {
    "committee","federal","reserve","monetary","policy","decided","decision",
    "meeting","target","rate","rates","longer","run","term","appropriate",
    "remain","remains","continue","continued","expects","anticipates","likely",
    "conditions","economic","activity","level","levels","pace","support",
    "stance","range","percent","action","board","approved","basis","available",
    "news","events","media","release","consistent","mandate","account", "chairman","announced","announce","today","believes"
}
LDA_STOPWORDS = set(ENGLISH_STOP_WORDS) | FOMC_BOILERPLATE

PHRASE_MAP = {
    r"\bfederal funds\b":"federal_funds", r"\blabor market\b":"labor_market",
    r"\bunemployment rate\b":"unemployment", r"\bprice pressures?\b":"price_pressure",
    r"\bjob gains\b":"job_gains", r"\bmortgage[- ]backed securities\b":"mbs",
    r"\btreasury securities\b":"treasury_securities",
    r"\basset purchases?\b":"asset_purchases", r"\bbalance sheet\b":"balance_sheet",
}

def lda_tokenize(text):
    text = normalize_text(strip_boilerplate(text))   # strip admin text FIRST
    for pat, tok in PHRASE_MAP.items():
        text = re.sub(pat, tok, text)
    words = re.findall(r"[a-z_]+", text)
    return [w for w in words if w not in LDA_STOPWORDS and len(w) > 2]

lda_docs = [" ".join(lda_tokenize(t)) for t in merged_finbert["statement_text"]]
print(f"Prepared {len(lda_docs)} statements for LDA.")

In [ ]:
# Vectorise -> document-term counts; scan K by UMass topic coherence; fit.
vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.4,
                             token_pattern=r"(?u)\b[a-z_]{3,}\b")
dtm = vectorizer.fit_transform(lda_docs)
vocab = vectorizer.get_feature_names_out()
print(f"Document-term matrix: {dtm.shape[0]} docs x {dtm.shape[1]} terms")

def umass_coherence(components, dtm, topn=10):
    """Self-contained UMass coherence (higher = more coherent). Replaces
    gensim's CoherenceModel so no extra dependency is needed."""
    B  = (dtm > 0).astype(int).tocsc()
    df = np.asarray(B.sum(axis=0)).ravel()
    out = []
    for comp in components:
        top = comp.argsort()[::-1][:topn]
        s, c = 0.0, 0
        for a in range(1, len(top)):
            for b in range(a):
                wi, wj = top[a], top[b]
                co = int(B[:, wi].multiply(B[:, wj]).sum())
                s += np.log((co + 1.0) / (df[wj] if df[wj] > 0 else 1.0)); c += 1
        out.append(s / max(c, 1))
    return float(np.mean(out))

print("\nUMass coherence scan (higher = better; pick K near the peak):")
for k in range(3, 9):
    _m = LatentDirichletAllocation(n_components=k, learning_method="batch",
                                   max_iter=25, random_state=42).fit(dtm)
    print(f"  K={k:2d}  umass={umass_coherence(_m.components_, dtm):.3f}")

K_LDA = 5   # <-- set from the scan above
lda_model = LatentDirichletAllocation(n_components=K_LDA, learning_method="batch",
                                      max_iter=50, random_state=42).fit(dtm)

print(f"\nTop words per topic (K={K_LDA}) - label these yourself:")
for t, comp in enumerate(lda_model.components_):
    print(f"  Topic {t}: " + ", ".join(vocab[i] for i in comp.argsort()[::-1][:10]))

In [ ]:
# Robustness: unigrams only (no artificial bigrams)
v1 = CountVectorizer(ngram_range=(1, 1), min_df=5, max_df=0.4,
                     token_pattern=r"(?u)\b[a-z_]{3,}\b")
dtm1 = v1.fit_transform(lda_docs)
vocab1 = v1.get_feature_names_out()
print("unigrams only:", dtm1.shape)

m1 = LatentDirichletAllocation(n_components=5, learning_method="batch",
                               max_iter=50, random_state=42).fit(dtm1)
for t, comp in enumerate(m1.components_):
    print(f"  Topic {t}: " + ", ".join(vocab1[i] for i in comp.argsort()[::-1][:10]))

In [ ]:
# Stage-1 output: per-meeting topic intensities theta_t (rows sum to 1).
theta = lda_model.transform(dtm)
theta = theta / theta.sum(axis=1, keepdims=True)     # ensure exact simplex
topic_cols = [f"topic_{t}" for t in range(K_LDA)]
theta_df = pd.DataFrame(theta, columns=topic_cols)

# merged_ext = the extended master frame (topics now; transformers added later)
merged_ext = pd.concat([merged_finbert.reset_index(drop=True), theta_df], axis=1)
merged_ext = merged_ext.sort_values("Date").reset_index(drop=True)
for c in topic_cols:
    merged_ext[f"d_{c}"] = merged_ext[c].diff()
d_topic_cols = [f"d_{c}" for c in topic_cols]

# attach topics to the HMT frame (which already carries the macro pre-state)
merged_ext_hmt = merged_hmt.merge(
    merged_ext[["Date"] + topic_cols + d_topic_cols], on="Date", how="left"
)
merged_ext[["Date"] + topic_cols].head()

In [ ]:
# Two small helpers used by every extension below.
def joint_p(model, vars_):
    """HC3 robust joint Wald p-value that all `vars_` coefficients are zero."""
    idx = {n: i for i, n in enumerate(model.params.index)}
    R = np.zeros((len(vars_), len(model.params)))
    for r, v in enumerate(vars_):
        R[r, idx[v]] = 1.0
    return float(model.wald_test(R, scalar=True).pvalue)

def dR2_over_mps(data, y, feat_cols):
    """Full (MPS+feats) model and its incremental R^2 over an MPS-only
    baseline fit on the SAME non-missing sample."""
    full = run_hc3_regression(data, y, ["MPS"] + feat_cols)
    sub  = data[[y, "MPS"] + feat_cols].dropna()
    base = sm.OLS(sub[y], sm.add_constant(sub[["MPS"]])).fit()
    return full, full.rsquared - base.rsquared

# Stage 2: asset responses on MPS + topic CHANGES (drop one topic as the
# reference, since the K intensities sum to 1).
lda_feats = d_topic_cols[:-1]
lda_results = {}
rows = []
for y in outcomes:
    full, d = dR2_over_mps(merged_ext, y, lda_feats)
    lda_results[y] = full
    rows.append({"outcome": y, "N": int(full.nobs),
                 "dR2_vs_MPS": round(d, 4),
                 "joint_p_topics": round(joint_p(full, lda_feats), 4)})
print("LDA topics: MPS + topic changes (one topic dropped as reference)\n")
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# Robustness: does the Stage-2 null hold across K?
for k in [3, 4, 5, 6, 7]:
    m = LatentDirichletAllocation(n_components=k, learning_method="batch",
                                  max_iter=50, random_state=42).fit(dtm)
    th = m.transform(dtm); th = th / th.sum(axis=1, keepdims=True)
    tmp = merged_finbert[["Date"]].reset_index(drop=True).copy()
    for j in range(k):
        tmp[f"tk{j}"] = th[:, j]
    tmp = tmp.sort_values("Date").reset_index(drop=True)
    dcols = []
    for j in range(k):
        tmp[f"d_tk{j}"] = tmp[f"tk{j}"].diff(); dcols.append(f"d_tk{j}")
    dcols = dcols[:-1]
    merged_k = merged_finbert.merge(tmp[["Date"] + dcols], on="Date", how="left")
    ps = []
    for y in outcomes:
        full, _ = dR2_over_mps(merged_k, y, dcols)
        ps.append(joint_p(full, dcols))
    ymin = outcomes[int(np.argmin(ps))]
    flag = "null holds" if min(ps) > 0.05 else "CHECK"
    print(f"K={k}: min joint p = {min(ps):.4f} at {ymin:15s} ({flag})")

In [ ]:
# HMT-style: residualise each topic LEVEL on the macro pre-state, then test.
for c in topic_cols:
    merged_ext_hmt[f"{c}_res"], _ = residualize(merged_ext_hmt, c, macro_vars)
topic_res_cols = [f"{c}_res" for c in topic_cols][:-1]   # drop one reference

lda_hmt_results = {}
rows = []
for y in outcomes_hmt:
    full, d = dR2_over_mps(merged_ext_hmt, y, topic_res_cols)
    lda_hmt_results[y] = full
    rows.append({"outcome": y, "N": int(full.nobs),
                 "dR2_vs_MPS": round(d, 4),
                 "joint_p": round(joint_p(full, topic_res_cols), 4)})
print("LDA residualised topics (HMT-style) + MPS:\n")
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# Orthogonality of topic changes to MPS (mirrors the BoW/FinBERT diagnostic).
print("Corr(topic change, MPS):")
for v in d_topic_cols:
    sub = merged_ext[[v, "MPS"]].dropna()
    print(f"  Corr({v:<10}, MPS) = {sub.corr().iloc[0,1]:+.4f}  (N={len(sub)})")


In [ ]:
# Robustness for the S&P 500 topic effect: statement-specific NEWS or persistent LEVELS?
# (a) residualise topic CHANGES on the macro pre-state, then test all outcomes
for c in d_topic_cols:
    merged_ext_hmt[f"{c}_res"], _ = residualize(merged_ext_hmt, c, macro_vars)
d_topic_res_cols = [f"{c}_res" for c in d_topic_cols][:-1]

print("HMT spec with residualised topic CHANGES (HF-consistent):")
rows = []
for y in outcomes_hmt:
    full, d = dR2_over_mps(merged_ext_hmt, y, d_topic_res_cols)
    rows.append({"outcome": y, "N": int(full.nobs),
                 "dR2_vs_MPS": round(d, 4),
                 "joint_p": round(joint_p(full, d_topic_res_cols), 4)})
print(pd.DataFrame(rows).to_string(index=False))

# (b) residualised topic LEVELS for the S&P 500, crisis periods excluded (GFC + COVID)
crisis = (((merged_ext_hmt["Date"] >= "2008-09-01") & (merged_ext_hmt["Date"] <= "2009-12-31")) |
          ((merged_ext_hmt["Date"] >= "2020-03-01") & (merged_ext_hmt["Date"] <= "2020-12-31")))
sub = merged_ext_hmt[~crisis]
full, d = dR2_over_mps(sub, "SP500_combined", topic_res_cols)
print(f"\nResidualised topic LEVELS, S&P 500, crisis excluded:  "
      f"N={int(full.nobs)}  dR2={d:.4f}  joint_p={joint_p(full, topic_res_cols):.4f}")

In [ ]:
sub = merged_ext_hmt.copy()
sub["post2008"] = (sub["Date"] >= "2008-12-16").astype(int)
full = run_hc3_regression(sub, "SP500_combined", ["MPS"] + topic_res_cols + ["post2008"])
print(f"S&P 500 ~ MPS + residualised topic levels + post2008 dummy:")
print(f"  joint_p(topics) = {joint_p(full, topic_res_cols):.4f}   (was 0.009 without the dummy)")

In [ ]:
# ======================================================================
# EXTENSION 2 - TRANSFORMER TEXT MEASURES  (run where huggingface.co is reachable)
#   A. roberta-base embeddings -> PCA      (unsupervised contextual content)
#   B. gtfintechlab/FOMC-RoBERTa           (hawkish/dovish STANCE)
#   C. Moritz-Pfeifer/CentralBankRoBERTa   (economic +/- SENTIMENT)
# Sentence-level (reusing split_into_sentences), aggregated per statement,
# then MPS-controlled Stage-2 in the SAME harness. Adds columns to merged_ext.
# Needs: transformers, torch (already imported for FinBERT), scikit-learn.
# ======================================================================
from transformers import (AutoTokenizer, AutoModel,
                          AutoModelForSequenceClassification)
from sklearn.decomposition import PCA


In [ ]:
# Hugging Face authentication is only required for gated models.
# gtfintechlab/FOMC-RoBERTa is access-gated; approval was not granted, so that
# measure is skipped. All other models used here are public.
# from huggingface_hub import login
# login()

In [ ]:
emb_tok = AutoTokenizer.from_pretrained("roberta-base")
emb_mdl = AutoModel.from_pretrained("roberta-base").eval()

def embed_statement(text):
    sents = split_into_sentences(text) or [str(text)]
    vecs = []
    for s in sents:
        enc = emb_tok(s, return_tensors="pt", truncation=True, max_length=256)
        with torch.no_grad():
            out = emb_mdl(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        vecs.append(((out * m).sum(1) / m.sum(1).clamp(min=1e-9)).cpu().numpy()[0])
    return np.mean(vecs, axis=0)

emb_mat = np.vstack([embed_statement(t) for t in tqdm(merged_ext["statement_text"])])
_X = (emb_mat - emb_mat.mean(0)) / (emb_mat.std(0) + 1e-9)
emb_pcs = PCA(n_components=5, random_state=42).fit_transform(_X)
emb_pc_cols = [f"emb_pc_{i}" for i in range(5)]
for i, c in enumerate(emb_pc_cols):
    merged_ext[c] = emb_pcs[:, i]
print("Added embedding PCs:", emb_pc_cols)

In [ ]:
cb_tok = AutoTokenizer.from_pretrained("Moritz-Pfeifer/CentralBankRoBERTa-sentiment-classifier")
cb_mdl = AutoModelForSequenceClassification.from_pretrained(
    "Moritz-Pfeifer/CentralBankRoBERTa-sentiment-classifier").eval()
_cb = {i: l.lower() for i, l in cb_mdl.config.id2label.items()}
_pos = next((i for i, l in _cb.items() if "pos" in l), None)
_neg = next((i for i, l in _cb.items() if "neg" in l), None)

def cb_roberta_sentiment(text):
    sents = split_into_sentences(text)
    if not sents:
        return np.nan
    net = []
    for s in sents:
        enc = cb_tok(s, return_tensors="pt", truncation=True, max_length=256)
        with torch.no_grad():
            p = torch.softmax(cb_mdl(**enc).logits, dim=1).cpu().numpy()[0]
        net.append(float((p[_pos] if _pos is not None else 0.0) -
                         (p[_neg] if _neg is not None else 0.0)))
    return float(np.mean(net))

merged_ext["CB_RoBERTa"] = merged_ext["statement_text"].progress_apply(cb_roberta_sentiment)
merged_ext = merged_ext.sort_values("Date").reset_index(drop=True)
merged_ext["d_CB_RoBERTa"] = merged_ext["CB_RoBERTa"].diff()

In [ ]:
_candidates = [
    ("CentralBankRoBERTa",  ["d_CB_RoBERTa"]),
    ("RoBERTa emb (PCA-5)", [f"emb_pc_{i}" for i in range(5)]),
]
for name, cols in _candidates:
    if not all(c in merged_ext.columns for c in cols):
        print(f"--- {name}: skipped ---\n"); continue
    rows = []
    for y in outcomes:
        full, d = dR2_over_mps(merged_ext, y, cols)
        rows.append({"outcome": y, "N": int(full.nobs),
                     "dR2_vs_MPS": round(d, 4), "joint_p": round(joint_p(full, cols), 4)})
    print(f"--- {name} ---"); print(pd.DataFrame(rows).to_string(index=False)); print()

In [ ]:
comparison_methods = {
    "BoW dictionary":      ["d_BoW"],
    "FinBERT":             ["d_FinBERT"],
    "LDA topics":          d_topic_cols[:-1],
    "CentralBankRoBERTa":  ["d_CB_RoBERTa"],
    "RoBERTa emb (PCA-5)": [f"emb_pc_{i}" for i in range(5)],
}
def _cell(data, y, cols):
    full, d = dR2_over_mps(data, y, cols)
    return f"{d:+.3f}{star(joint_p(full, cols))}"
cmp_rows = []
for name, cols in comparison_methods.items():
    if not all(c in merged_ext.columns for c in cols): continue
    cmp_rows.append({"method": name, **{y: _cell(merged_ext, y, cols) for y in outcomes}})
comparison_table = pd.DataFrame(cmp_rows).set_index("method")
print("Incremental R^2 over MPS-only baseline (stars = joint p-value):\n")
print(comparison_table.to_string())

In [ ]:
me = merged_ext.sort_values("Date").reset_index(drop=True)
for c in emb_pc_cols:
    me[f"d_{c}"] = me[c].diff()
d_emb_cols = [f"d_{c}" for c in emb_pc_cols]
print("RoBERTa embedding CHANGES (HF-consistent):")
for y in ["SP500_combined","SP500","SP500_emini","SLOPE_10_2","TNOTE02"]:
    full, d = dR2_over_mps(me, y, d_emb_cols)
    print(f"  {y:15s} dR2={d:.4f}  joint_p={joint_p(full, d_emb_cols):.4f}")
merged_ext = me

In [ ]:
sub = merged_ext.copy()
sub["post2008"] = (sub["Date"] >= "2008-12-16").astype(int)
full = run_hc3_regression(sub, "SP500_combined", ["MPS"] + emb_pc_cols + ["post2008"])
print(f"emb levels + post2008 dummy: joint_p = {joint_p(full, emb_pc_cols):.4f}  (was 0.0031)")

In [ ]:
yr = merged_ext["Date"].dt.year
for c in emb_pc_cols:
    print(f"  Corr({c}, year) = {merged_ext[c].corr(yr):+.3f}")

In [ ]:
# (a) does the slope effect survive the era dummy too?
sub = merged_ext.copy()
sub["post2008"] = (sub["Date"] >= "2008-12-16").astype(int)
full = run_hc3_regression(sub, "SLOPE_10_2", ["MPS"] + d_emb_cols + ["post2008"])
print(f"SLOPE_10_2 changes + post2008: joint_p = {joint_p(full, d_emb_cols):.4f}  (was 0.009)")

# (b) is it just the 5-PC flexibility? refit with 3 PCs
full3, _ = dR2_over_mps(merged_ext, "SLOPE_10_2", d_emb_cols[:3])
print(f"SLOPE_10_2 changes, 3 PCs only: joint_p = {joint_p(full3, d_emb_cols[:3]):.4f}")

In [ ]:
# 1. WHICH PC carries it? (interpretability - is it one direction or diffuse?)
for i in range(5):
    full1, _ = dR2_over_mps(merged_ext, "SLOPE_10_2", [f"d_emb_pc_{i}"])
    print(f"  d_emb_pc_{i} alone: joint_p = {joint_p(full1, [f'd_emb_pc_{i}']):.4f}")

# 2. Does it hold with HC3 on the OTHER two slopes' changes? (is 10-2 special or lucky?)
for y in ["SLOPE_5_2","SLOPE_10_5"]:
    full, d = dR2_over_mps(merged_ext, y, d_emb_cols)
    print(f"  {y}: dR2={d:.4f} joint_p={joint_p(full, d_emb_cols):.4f}")

# 3. Newey-West / does it survive excluding the ZLB period? (not a level effect in disguise)
sub = merged_ext[(merged_ext['Date'] < '2008-12-16') | (merged_ext['Date'] > '2015-12-16')]
full, d = dR2_over_mps(sub, "SLOPE_10_2", d_emb_cols)
print(f"  SLOPE_10_2 ex-ZLB: N={int(full.nobs)} dR2={d:.4f} joint_p={joint_p(full, d_emb_cols):.4f}")

In [ ]:
# What does PC2 represent? Read the extreme statements on that axis.
tmp = merged_ext[["Date","emb_pc_2"]].copy()
print("MOST NEGATIVE on PC2:"); print(tmp.nsmallest(5,"emb_pc_2").to_string(index=False))
print("\nMOST POSITIVE on PC2:"); print(tmp.nlargest(5,"emb_pc_2").to_string(index=False))

In [ ]:
# Drop the most extreme crisis statements and see if the slope effect survives
extreme = merged_ext["emb_pc_2"] > 12   # the 2007-08 / 2008 cluster
sub = merged_ext[~extreme]
full, d = dR2_over_mps(sub, "SLOPE_10_2", d_emb_cols)
print(f"SLOPE_10_2 changes, dropping acute-crisis statements: "
      f"N={int(full.nobs)} dR2={d:.4f} joint_p={joint_p(full, d_emb_cols):.4f}")